<a href="https://colab.research.google.com/github/jeffersonpardo00/MIT107-Property-Maintenance-Management-/blob/main/Property_Maintenance_Management_System.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import sqlite3
import pandas as pd
# Connect to an in-memory SQLite database (perfect for prototyping)
# If you want to save the database as a file, change ':memory:' to 'my_database.db'
conn = sqlite3.connect(':memory:')
cursor = conn.cursor()
print("Database connection established successfully.")

Database connection established successfully.


In [ ]:
# Connect to the SQLite database
connection = sqlite3.connect("property_management.db")
cursor = connection.cursor()

# Enable foreign key constraints # hola
cursor.execute("PRAGMA foreign_keys = ON")

# 1. Create Users table
cursor.execute("""
CREATE TABLE IF NOT EXISTS Users (
    user_id INTEGER PRIMARY KEY AUTOINCREMENT,
    full_name TEXT NOT NULL,
    email TEXT NOT NULL UNIQUE,
    password TEXT NOT NULL,
    phone_number TEXT,
    role TEXT NOT NULL CHECK (role IN ('Tenant', 'Agent'))
)
""")

# 2. Create TicketStatus table
cursor.execute("""
CREATE TABLE IF NOT EXISTS TicketStatus (
    status_id INTEGER PRIMARY KEY AUTOINCREMENT,
    status_name TEXT NOT NULL UNIQUE
)
""")

# 3. Create MaintenanceTicket table
cursor.execute("""
CREATE TABLE IF NOT EXISTS MaintenanceTicket (
    ticket_id INTEGER PRIMARY KEY AUTOINCREMENT,
    tenant_id INTEGER NOT NULL,
    agent_id INTEGER,
    description TEXT NOT NULL,
    priority TEXT NOT NULL CHECK (priority IN ('Low', 'Medium', 'High')),
    date_created TEXT NOT NULL,
    last_updated TEXT NOT NULL,
    status_id INTEGER NOT NULL,

    FOREIGN KEY (tenant_id) REFERENCES Users(user_id),
    FOREIGN KEY (agent_id) REFERENCES Users(user_id),
    FOREIGN KEY (status_id) REFERENCES TicketStatus(status_id)
)
""")

# 4. Insert dummy data into Users
users = [
    ("John Smith", "john.smith@email.com", "password123", "0412345678", "Tenant"),
    ("Sarah Johnson", "sarah.johnson@email.com", "password456", "0423456789", "Tenant"),
    ("Michael Brown", "michael.brown@email.com", "password789", "0434567890", "Agent"),
    ("Emily Davis", "emily.davis@email.com", "password321", "0445678901", "Agent")
]

cursor.executemany("""
INSERT INTO Users (full_name, email, password, phone_number, role)
VALUES (?, ?, ?, ?, ?)
""", users)

# 5. Insert dummy data into TicketStatus
statuses = [
    ("OPEN",),
    ("IN_PROGRESS",),
    ("RESOLVED",)
]

cursor.executemany("""
INSERT INTO TicketStatus (status_name)
VALUES (?)
""", statuses)

# 6. Insert dummy data into MaintenanceTicket
tickets = [
    (1, 3, "Leaking kitchen tap requires repair", "Medium",
     "2026-11-10", "2026-11-10", 1),

    (2, 3, "Heating system is not working", "High",
     "2026-11-11", "2026-11-12", 2),

    (1, 4, "Broken bathroom light fixture", "Low",
     "2026-11-13", "2026-11-14", 3)
]

cursor.executemany("""
INSERT INTO MaintenanceTicket (
    tenant_id,
    agent_id,
    description,
    priority,
    date_created,
    last_updated,
    status_id
)
VALUES (?, ?, ?, ?, ?, ?, ?)
""", tickets)

# 7. Execute SELECT query joining MaintenanceTicket and Users
cursor.execute("""
SELECT
    MaintenanceTicket.ticket_id,
    Users.full_name AS tenant_name,
    MaintenanceTicket.description,
    MaintenanceTicket.priority,
    TicketStatus.status_name
FROM MaintenanceTicket
INNER JOIN Users
    ON MaintenanceTicket.tenant_id = Users.user_id
INNER JOIN TicketStatus
    ON MaintenanceTicket.status_id = TicketStatus.status_id
WHERE Users.role = 'Tenant'
""")

# 8. Display query results
print("Maintenance Tickets:")
print("-" * 80)

for row in cursor.fetchall():
    print(f"Ticket ID: {row[0]}")
    print(f"Tenant: {row[1]}")
    print(f"Description: {row[2]}")
    print(f"Priority: {row[3]}")
    print(f"Status: {row[4]}")
    print("-" * 80)

# 9. Commit changes and close connection
connection.commit()
connection.close()

print("Database created and populated successfully.")

Maintenance Tickets:
--------------------------------------------------------------------------------
Ticket ID: 1
Tenant: John Smith
Description: Leaking kitchen tap requires repair
Priority: Medium
Status: OPEN
--------------------------------------------------------------------------------
Ticket ID: 2
Tenant: Sarah Johnson
Description: Heating system is not working
Priority: High
Status: IN_PROGRESS
--------------------------------------------------------------------------------
Ticket ID: 3
Tenant: John Smith
Description: Broken bathroom light fixture
Priority: Low
Status: RESOLVED
--------------------------------------------------------------------------------
Database created and populated successfully.
